# Meja Kerja AR-2026-082 — Laporan Siap Keputusan

## Persiapan

In [1]:
from pathlib import Path
import os, sys, csv, json, subprocess

# Cari folder utama proyek, baik notebook dibuka dari folder utama maupun folder notebooks.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir() and (PROJECT_ROOT.parent / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Folder src tidak ditemukan. Buka notebook dari dalam folder proyek.')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import read_metrics, read_content

rows = read_metrics('data/input/metrics.csv')
print('Folder proyek:', PROJECT_ROOT)
print('Jumlah baris:', len(rows))
print(rows[:3])


Folder proyek: C:\Users\farich as\Downloads\bootcamp advance\prabootcamp advance\FSB_Decision_Ready_Report_Delivery_Lead_Pack\FSB_Decision_Ready_Report_Delivery_Lead_Pack_Bahasa_Indonesia
Jumlah baris: 25
[{'metric': 'total_transactions', 'rank': '1', 'dimension': '', 'value': '22'}, {'metric': 'total_revenue', 'rank': '1', 'dimension': '', 'value': '8745000'}, {'metric': 'aov', 'rank': '1', 'dimension': '', 'value': '397500'}]


## Pencarian metrik

In [2]:
lookup={(r['metric'],r['dimension']):float(r['value']) for r in rows}
for k in [('total_revenue',''),('category_revenue','Electronics'),('city_revenue','Unknown')]:print(k,lookup[k])

('total_revenue', '') 8745000.0
('category_revenue', 'Electronics') 8550000.0
('city_revenue', 'Unknown') 900000.0


## Perhitungan bukti

In [3]:
share=lookup[('category_revenue','Electronics')]/lookup[('total_revenue','')];print(share, f'{share*100:.1f}%')

0.9777015437392796 97.8%


Tulis temuan, makna, tindakan, dan batasan sebelum membuat kode.

## Periksa kontrak isi laporan

In [4]:
content=read_content('content/report_content.json');print(json.dumps(content,indent=2,ensure_ascii=False))

{
  "title": "Laporan Penjualan Siap Keputusan",
  "scope": "Data uji gerbang: 22 transaksi valid; hanya gambaran sesaat",
  "insights": [
    {
      "title": "Pendapatan sangat terkonsentrasi pada Electronics",
      "evidence": "Electronics menghasilkan Rp8.550.000 dari total Rp8.745.000 atau 97,8%.",
      "meaning": "Total pendapatan sensitif terhadap satu kategori.",
      "action": "Validasi tren periode lain dan periksa margin sebelum mengambil keputusan persediaan.",
      "boundary": "Data uji berukuran kecil serta tidak memiliki tanggal transaksi atau margin."
    },
    {
      "title": "Webcam mendominasi jumlah unit",
      "evidence": "Webcam terjual sebanyak 19 dari total 22 unit.",
      "meaning": "Permintaan pada data uji terkonsentrasi pada satu produk.",
      "action": "Periksa persediaan, waktu tunggu, dan konsistensi permintaan pada periode lain.",
      "boundary": "Jumlah unit yang tinggi tidak membuktikan laba tertinggi."
    },
    {
      "title": "Analisis

## Jalankan pembuat laporan

In [5]:
r=subprocess.run([sys.executable,'generate_report.py','--metrics','data/input/metrics.csv','--content','content/report_content.json','--output','data/output/report.md'],text=True,capture_output=True);print(r.returncode,r.stdout,r.stderr)

0 REPORT READY path=data/output/report.md chars=2544
 


## Pratinjau laporan

In [6]:
p=Path('data/output/report.md');print(p.read_text() if p.exists() else 'MISSING')

# Laporan Penjualan Siap Keputusan

**Cakupan:** Data uji gerbang: 22 transaksi valid; hanya gambaran sesaat

## Ringkasan Eksekutif
- Pendapatan tercatat Rp8.745.000 dari 22 transaksi.
- Nilai transaksi rata-rata adalah Rp397.500; pembeli unik 14; pembeli berulang 8.
- Electronics menyumbang 97,8% pendapatan pada data uji ini.

## Ringkasan Metrik
| Metrik | Nilai | Kunci sumber |
|---|---:|---|
| Total transaksi | 22 | total_transactions |
| Total pendapatan | Rp8.745.000 | total_revenue |
| Nilai transaksi rata-rata | Rp397.500 | aov |
| Pembeli unik | 14 | unique_buyers |
| Pembeli berulang | 8 | repeat_buyers |
| Porsi Electronics | 97,8% | category_revenue/Electronics |
| Pendapatan kota Unknown | Rp900.000 | city_revenue/Unknown |

## Temuan
### 1. Pendapatan sangat terkonsentrasi pada Electronics
- **Bukti:** Electronics menghasilkan Rp8.550.000 dari total Rp8.745.000 atau 97,8%.
- **Makna:** Total pendapatan sensitif terhadap satu kategori.
- **Tindakan:** Validasi tren period

## Telusuri satu pernyataan

In [7]:
print([r for r in rows if r['metric']=='category_revenue'])

[{'metric': 'category_revenue', 'rank': '1', 'dimension': 'Electronics', 'value': '8550000'}, {'metric': 'category_revenue', 'rank': '2', 'dimension': 'Home Office', 'value': '125000'}, {'metric': 'category_revenue', 'rank': '3', 'dimension': 'Stationery', 'value': '70000'}]


## Pengujian: masukan berubah

In [8]:
r=subprocess.run([sys.executable,'generate_report.py','--metrics','data/failure_fixtures/changed_revenue.csv','--content','content/report_content.json','--output','data/output/changed_report.md'],text=True,capture_output=True);print(r.returncode,r.stdout,r.stderr)

0 REPORT READY path=data/output/changed_report.md chars=2544
 


Bandingkan pendapatan pada laporan awal dan laporan setelah perubahan. Perubahannya harus merambat.

## Pengujian: metrik hilang

In [9]:
r=subprocess.run([sys.executable,'generate_report.py','--metrics','data/failure_fixtures/missing_aov.csv','--content','content/report_content.json','--output','data/output/bad.md'],text=True,capture_output=True);print(r.returncode,r.stderr)

1 Traceback (most recent call last):
  File "C:\Users\farich as\Downloads\bootcamp advance\prabootcamp advance\FSB_Decision_Ready_Report_Delivery_Lead_Pack\FSB_Decision_Ready_Report_Delivery_Lead_Pack_Bahasa_Indonesia\generate_report.py", line 8, in <module>
    if __name__=='__main__':raise SystemExit(main())
                                             ^^^^^^
  File "C:\Users\farich as\Downloads\bootcamp advance\prabootcamp advance\FSB_Decision_Ready_Report_Delivery_Lead_Pack\FSB_Decision_Ready_Report_Delivery_Lead_Pack_Bahasa_Indonesia\generate_report.py", line 7, in main
    rows=read_metrics(a.metrics);c=read_content(a.content);validate_rows(rows);validate_content(c);text=render_report(rows,c);write_text(a.output,text);print(f'REPORT READY path={a.output} chars={len(text)}');return 0
                                                           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\farich as\Downloads\bootcamp advance\prabootcamp advance\FSB_Decision_Ready_Report_Delivery_Lead_Pack\FS

## Pengujian: kunci ganda

In [10]:
r=subprocess.run([sys.executable,'generate_report.py','--metrics','data/failure_fixtures/duplicate_key.csv','--content','content/report_content.json','--output','data/output/bad.md'],text=True,capture_output=True);print(r.returncode,r.stderr)

1 Traceback (most recent call last):
  File "C:\Users\farich as\Downloads\bootcamp advance\prabootcamp advance\FSB_Decision_Ready_Report_Delivery_Lead_Pack\FSB_Decision_Ready_Report_Delivery_Lead_Pack_Bahasa_Indonesia\generate_report.py", line 8, in <module>
    if __name__=='__main__':raise SystemExit(main())
                                             ^^^^^^
  File "C:\Users\farich as\Downloads\bootcamp advance\prabootcamp advance\FSB_Decision_Ready_Report_Delivery_Lead_Pack\FSB_Decision_Ready_Report_Delivery_Lead_Pack_Bahasa_Indonesia\generate_report.py", line 7, in main
    rows=read_metrics(a.metrics);c=read_content(a.content);validate_rows(rows);validate_content(c);text=render_report(rows,c);write_text(a.output,text);print(f'REPORT READY path={a.output} chars={len(text)}');return 0
                                                           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\farich as\Downloads\bootcamp advance\prabootcamp advance\FSB_Decision_Ready_Report_Delivery_Lead_Pack\FS

## Tinjauan batasan
Jelaskan mengapa pendapatan tidak dapat membuktikan laba atau hubungan sebab-akibat.

## Pilihan pekerjaan lanjutan
Setelah pekerjaan utama selesai, pilih satu pengembangan lanjutan yang sesuai.

## Pemeriksaan akhir mandiri

In [11]:
r=subprocess.run([sys.executable,'delivery_check.py'],text=True,capture_output=True);print(r.stdout,r.stderr)

01_input_smoke.py PASS 25
02_metrics_contract.py PASS contract
03_report_acceptance.py PASS
04_traceability.py PASS ['total_revenue', 'category_revenue/Electronics', 'city_revenue/Unknown']
05_failure_controls.py EXPECTED missing_aov.csv missing_required_metrics=[('aov', '')]
EXPECTED duplicate_key.csv duplicate_metric_key=('total_transactions', '')
EXPECTED non_numeric.csv non_numeric_value row=3 value=Rp8jt
PASS
06_delivery_integrity.py PASS delivery integrity
READY FOR REVIEW
 


## Ringkasan penutup
Sampaikan satu temuan, satu batasan, dan satu jalur ketertelusuran lengkap.